# Pipeline Deployment & YAML Orchestration
**NB 22** | End-to-end pipeline automation with scheduling

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)


In [ ]:
# Pipeline deployment — define a recurring monitoring workflow
BBOX       = (-0.15, 51.47, -0.10, 51.52)   # London — same AOI as NB01
DATA_DIR   = pathlib.Path('./results/nb22')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Method 1: Python chainable pipeline builder (PyGeoFetch v2.0 / stub fallback)
chain = (
    client.pgf_pipeline('london-weekly-monitoring')
    .atmos(method='dos1')
    .cloud_mask(method='scl')
    .clip(bbox=BBOX)
    .reproject(crs='EPSG:32630')
    .ndvi(red='B04.tif', nir='B08.tif')
    .cog(compress='deflate')
)
print(f"Pipeline chain configured: {len(chain._steps)} steps")
for i, (op, kw) in enumerate(chain._steps, 1):
    print(f"  Step {i}: {op}({', '.join(f'{k}={v!r}' for k,v in kw.items())})")


In [ ]:
# Method 2: Batch processing — same preprocessing chain on multiple scenes
results = client.search(bbox=BBOX, date_range=('2024-01-01','2024-12-31'),
    providers=['planetary_computer'], cloud_cover_max=10)
downloads = client.download(results[:3], output_dir=str(DATA_DIR/'batch'),
    bands=['B02','B03','B04','B08','B11','B12'], post_process=['reproject:EPSG:32630','cog'])
scene_paths = [str(d.path) for d in downloads if d.success]
print(f"Downloaded {len(scene_paths)} scenes for batch processing")


In [ ]:
# Validate ALL scenes before batch preprocessing
for sp in scene_paths:
    rep = client.validator.validate(sp, required_bands=6)
    print(f"  {pathlib.Path(sp).name[:40]:<40} : {'PASSED' if rep.passed else 'FIXED'}")


In [ ]:
# Batch preprocess all scenes through the full pipeline
BATCH_OUT = DATA_DIR / 'batch_preprocessed'
BATCH_OUT.mkdir(exist_ok=True)
batch_results = client.batch_process(
    inputs     = scene_paths,
    chain      = [
        ('clip',      {'bbox': BBOX}),
        ('normalise', {'method': 'scale_factor', 'scale_factor': 10000.0}),
        ('cog',       {}),
    ],
    output_dir = str(BATCH_OUT),
    parallel   = 4,
)
print("Batch preprocessing results:")
for r in batch_results:
    status = "✓" if r['success'] else "✗"
    print(f"  {status} {pathlib.Path(r['input']).name[:40]:<40} → {pathlib.Path(r['output']).name}")


In [ ]:
# Method 3: Named E2E pipeline (search + download + preprocess + model + postprocess)
pipeline_result = client.pipeline(
    'building_footprints',
    bbox       = BBOX,
    date       = '2024-06',
    output_dir = str(DATA_DIR/'pipeline_output'),
)
print(f"Pipeline result: {pipeline_result}")


In [ ]:
# Deployment summary
print('='*60)
print('Pipeline Deployment Summary')
print('='*60)
print(f"  Chain steps configured : {len(chain._steps)}")
print(f"  Scenes batch processed : {len(batch_results)}")
print(f"  Success rate           : {sum(1 for r in batch_results if r['success'])}/{len(batch_results)}")
print("  51 named pipelines     : client.pipeline('NAME', bbox=..., date=...)")
print("  Chainable builder      : client.pgf_pipeline('name').clip().ndvi().cog().run(...)")
print("  Batch processing       : client.batch_process(inputs, chain, output_dir)")


**NB22 — Pipeline Deployment** complete.
- Study area: London UK
- Sensor: Sentinel-2
- Model: Chainable pipeline + batch processing
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG